# Stellar-Qwen3-8B-ARM — RewardBench 2 Evaluation

- **Model:** [`Stellar-lab/Stellar-Qwen3-8B-ARM`](https://huggingface.co/Stellar-lab/Stellar-Qwen3-8B-ARM)
- **Dataset:** [allenai/reward-bench-2](https://huggingface.co/datasets/allenai/reward-bench-2) (1,865 prompts / 8,977 completions across 6 domains)
- **Reported Result:** **74.20%** Overall Macro-Average (Random Baseline: `25.00%`)
- **Hardware:** Requires a CUDA GPU (`>=2` GPUs run FP16 via `device_map="auto"`; `1` GPU runs 4-bit NF4 quantization)

In [1]:
# ==============================================================================
# CELL 0: Dependencies Installation & Environment Setup
# ==============================================================================
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    "accelerate>=0.33.0" \
    "transformers>=4.44.0" \
    "datasets>=2.20.0" \
    "tiktoken" \
    "sentencepiece" \
    "seaborn" \
    "scipy" \
    "tabulate"

import sys
import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("=" * 80)
print("REWARDBENCH 2 EVALUATION ENVIRONMENT")
print("=" * 80)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers:         {transformers.__version__}")
print(f"BitsAndBytes:         {bnb.__version__}")
print(f"Accelerate:           {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()} ({torch.cuda.device_count()} GPU(s))")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GiB")
print("=" * 80)

REWARDBENCH 2 EVALUATION ENVIRONMENT
Python Version:       3.13.16
PyTorch Version:      2.11.0+cu130 (CUDA: 13.0)
Transformers:         5.22.2
BitsAndBytes:         0.50.2
Accelerate:           1.15.0
CUDA Available:       True (2 GPU(s))
  GPU 0: Tesla T4 | VRAM: 14.56 GiB
  GPU 1: Tesla T4 | VRAM: 14.56 GiB


In [2]:
# ==============================================================================
# CELL 1: Load Stellar-Qwen3-8B-ARM from Hugging Face Hub
# ==============================================================================
import os
import sys
import gc
import math
import types
import torch
import transformers.modeling_utils
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Stellar-lab/Stellar-Qwen3-8B-ARM"
print(f"[MODEL] Loading model from Hugging Face Hub: {MODEL_ID}")

class TiedWeightsDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, "_all_tied_weights_keys_storage", {})
    def __set__(self, instance, value):
        instance._all_tied_weights_keys_storage = value

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = TiedWeightsDescriptor()
transformers.modeling_utils.PreTrainedModel._tied_weights_keys = {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"
print(f"[TOKENIZER] Vocab size: {len(tokenizer):,} | Pad Token ID: {tokenizer.pad_token_id}")

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    print(f"[HARDWARE] Loading FP16 model partitioned across {num_gpus} GPUs...")
    max_memory = {i: "13.0GiB" for i in range(num_gpus)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
    )
elif num_gpus == 1:
    print("[HARDWARE] Single GPU detected -> Loading in 4-bit NF4...")
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    raise RuntimeError("A CUDA GPU is required for evaluation.")

def multi_gpu_forward_dag_head_only(self, last_6_hidden: torch.Tensor):
    head_dev = self.dag_down_proj.weight.device
    head_dtype = self.dag_down_proj.weight.dtype
    h_in = last_6_hidden.to(device=head_dev, dtype=head_dtype)

    z = self.dag_norm(self.dag_down_proj(h_in))
    for layer in self.dag_layers:
        z = layer(z, associative=self.config.associative)

    raw_logits = []
    for i in range(self.config.num_dimensions):
        head_i = self.dim_heads[i]
        h_dim = z[:, i, :].to(device=head_i.weight.device, dtype=head_i.weight.dtype)
        raw_logits.append(head_i(h_dim).to(device=head_dev))

    raw_z = torch.cat(raw_logits, dim=-1).float()
    z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)
    agg_weights = torch.nn.functional.softmax(self.agg_logits.to(device=head_dev).float(), dim=0)

    if self.config.scalar_only_mode:
        dim_logits = z_clamped
        dim_scores = torch.sigmoid(dim_logits)
        scalar_logit = (dim_logits * agg_weights).sum(dim=-1)
        return {
            "scalar_logit": scalar_logit,
            "dim_logits": dim_logits,
            "dim_scores": dim_scores,
            "agg_weights": agg_weights,
            "veto_penalty": torch.zeros_like(scalar_logit),
        }

    if self.config.associative:
        z3 = z_clamped[:, 2:3]
        z4 = z_clamped[:, 3:4]
        tau_m = self.config.monotonic_tau
        z_prereq = (
            -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True)
            + tau_m * math.log(2.0)
        )
        z5_mono = torch.clamp(
            z_prereq + self.config.monotonic_delta_tol - torch.nn.functional.softplus(-raw_z[:, 4:5]),
            min=-15.0,
            max=15.0,
        )
        dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
    else:
        dim_logits = z_clamped

    dim_scores = torch.sigmoid(dim_logits)
    linear_term = self.config.beta_lin * (dim_logits * agg_weights).sum(dim=-1)

    gk_logits = dim_logits[:, self.config.gatekeeper_indices]
    kappa = self.config.veto_kappa
    veto_lambda = torch.nn.functional.softplus(self.veto_raw_lambda.to(device=head_dev).float())
    veto_thresh = self.veto_thresholds.to(device=head_dev).float().unsqueeze(0)

    exponents = -kappa * (gk_logits - veto_thresh)
    zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=head_dev)
    veto_penalty = (veto_lambda / kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)

    scalar_logit = linear_term - veto_penalty
    return {
        "scalar_logit": scalar_logit,
        "dim_logits": dim_logits,
        "dim_scores": dim_scores,
        "agg_weights": agg_weights,
        "veto_penalty": veto_penalty,
    }

model.forward_dag_head_only = types.MethodType(multi_gpu_forward_dag_head_only, model)
model.eval()
input_device = model.backbone.get_input_embeddings().weight.device

print("=" * 80)
print(f"[MODEL READY] Loaded {model.__class__.__name__}")
for i in range(num_gpus):
    vram_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  GPU {i} Allocated VRAM: {vram_gb:.2f} GB / {total_gb:.2f} GB")
print("=" * 80)

[MODEL READY] Loaded StellarQwen3ARMForRewardModeling
  GPU 0 Allocated VRAM: 7.99 GB / 14.56 GB
  GPU 1 Allocated VRAM: 6.14 GB / 14.56 GB


In [3]:
# ==============================================================================
# CELL 2: Load Official `allenai/reward-bench-2` Dataset
# ==============================================================================
import os
import numpy as np
import pandas as pd
from datasets import load_dataset

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_PATH = os.path.join(OUTPUT_DIR, "rewardbench2_predictions.parquet")
LEADERBOARD_CSV_PATH = os.path.join(OUTPUT_DIR, "rewardbench2_leaderboard.csv")
LEADERBOARD_PNG_PATH = os.path.join(OUTPUT_DIR, "rewardbench2_comparison.png")

print("[DATASET] Loading official `allenai/reward-bench-2` from Hugging Face...")
try:
    ds_rb2 = load_dataset("allenai/reward-bench-2", split="test")
    df_rb2 = ds_rb2.to_pandas()
except Exception:
    ds_all = load_dataset("allenai/reward-bench-2")
    split_key = list(ds_all.keys())[0]
    df_rb2 = ds_all[split_key].to_pandas()

total_completions_count = int(df_rb2["total_completions"].sum()) if "total_completions" in df_rb2.columns else 0
print("=" * 80)
print(f"[REWARDBENCH 2 LOADED] Total Prompts: {len(df_rb2):,} | Total Completions: {total_completions_count:,}")
print("-" * 80)
for subset, grp in df_rb2.groupby("subset"):
    n_p = len(grp)
    n_c = int(grp["total_completions"].sum()) if "total_completions" in grp.columns else 0
    print(f"  - {subset:<20s}: {n_p:>4d} prompts ({n_c:>5d} total completions)")
print("=" * 80)

[DATASET] Loading official `allenai/reward-bench-2` from Hugging Face...


[REWARDBENCH 2 LOADED] Total Prompts: 1,865 | Total Completions: 8,977
--------------------------------------------------------------------------------
  - Factuality          :  475 prompts ( 1900 total completions)
  - Focus               :  495 prompts ( 1980 total completions)
  - Math                :  183 prompts (  732 total completions)
  - Precise IF          :  160 prompts (  640 total completions)
  - Safety              :  450 prompts ( 1800 total completions)
  - Ties                :  102 prompts ( 1925 total completions)


In [4]:
# ==============================================================================
# CELL 3: Memory-Safe Inference Across All 1,865 RewardBench 2 Prompts
# ==============================================================================
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict, Any

MAX_LEN = 1280
RESP_RESERVE = 896
MICRO_BATCH_SEQS = 1  # Single-sequence forward pass prevents 4D attention mask OOM on 2x T4 GPUs

def encode_prompt_completion_tail_safe(prompt_str: str, resp_str: str) -> List[int]:
    p_clean = str(prompt_str).strip()
    if not p_clean.startswith("User:"):
        p_clean = f"User: {p_clean}\nAssistant: "
    p_ids = tokenizer.encode(p_clean, add_special_tokens=False)
    r_ids = tokenizer.encode(str(resp_str).strip(), add_special_tokens=False)

    if len(p_ids) + len(r_ids) <= MAX_LEN:
        return p_ids + r_ids

    r_keep = r_ids[-min(len(r_ids), RESP_RESERVE):]
    p_budget = MAX_LEN - len(r_keep)
    p_keep = p_ids[-p_budget:] if p_budget > 0 else []
    return p_keep + r_keep

@torch.no_grad()
def score_completions_for_prompt(prompt_str: str, completions: List[str]) -> List[Dict[str, Any]]:
    token_lists = [encode_prompt_completion_tail_safe(prompt_str, c) for c in completions]
    pad_id = tokenizer.pad_token_id
    out_records = []

    for start_idx in range(0, len(token_lists), MICRO_BATCH_SEQS):
        sub_lists = token_lists[start_idx : start_idx + MICRO_BATCH_SEQS]
        max_l = max(len(x) for x in sub_lists)
        b_sz = len(sub_lists)

        padded = torch.full((b_sz, max_l), pad_id, dtype=torch.long, device=input_device)
        mask = torch.zeros((b_sz, max_l), dtype=torch.long, device=input_device)
        for i, ids in enumerate(sub_lists):
            pad_len = max_l - len(ids)
            padded[i, pad_len:] = torch.tensor(ids, dtype=torch.long, device=input_device)
            mask[i, pad_len:] = 1

        out = model(input_ids=padded, attention_mask=mask)
        logits_np = out.logits.float().cpu().numpy()
        veto_np = out.veto_penalty.float().cpu().numpy()
        dims_np = out.dim_scores.float().cpu().numpy()

        for i in range(b_sz):
            out_records.append({
                "score": float(logits_np[i]),
                "veto": float(veto_np[i]),
                "d1_adh": float(dims_np[i, 0]),
                "d4_reas": float(dims_np[i, 3]),
                "d5_corr": float(dims_np[i, 4]),
            })
        del padded, mask, out
    return out_records

print(f"[INFERENCE] Evaluating {len(df_rb2):,} RewardBench 2 prompts...")
eval_rows = []
records = df_rb2.to_dict("records")

for row_idx, r in enumerate(tqdm(records, desc="RewardBench 2")):
    prompt_txt = r["prompt"]
    chosen_list = list(r["chosen"]) if isinstance(r["chosen"], (list, np.ndarray)) else [str(r["chosen"])]
    rejected_list = list(r["rejected"]) if isinstance(r["rejected"], (list, np.ndarray)) else [str(r["rejected"])]

    all_completions = chosen_list + rejected_list
    all_scored = score_completions_for_prompt(prompt_txt, all_completions)

    n_c = len(chosen_list)
    c_scored = all_scored[:n_c]
    r_scored = all_scored[n_c:]

    c_scores = [x["score"] for x in c_scored]
    r_scores = [x["score"] for x in r_scored]

    eval_rows.append({
        "id": str(r.get("id", row_idx)),
        "subset": str(r["subset"]),
        "num_correct": int(n_c),
        "num_rejected": int(len(r_scores)),
        "chosen_scores": c_scores,
        "rejected_scores": r_scores,
        "max_chosen_score": float(max(c_scores)),
        "min_chosen_score": float(min(c_scores)),
        "max_rejected_score": float(max(r_scores)),
        "mean_chosen_veto": float(np.mean([x["veto"] for x in c_scored])),
        "mean_rejected_veto": float(np.mean([x["veto"] for x in r_scored])),
    })

    if (row_idx + 1) % 100 == 0:
        torch.cuda.empty_cache()

df_rb2_preds = pd.DataFrame(eval_rows)
df_rb2_preds.to_parquet(PREDICTIONS_PATH, index=False)
print(f"\n[SAVED] All {len(df_rb2_preds):,} RewardBench 2 predictions saved to: {PREDICTIONS_PATH}")

[SAVED] All 1,865 RewardBench 2 predictions saved to: /kaggle/working/rewardbench2_predictions.parquet


In [5]:
# ==============================================================================
# CELL 4: Official AllenAI RewardBench 2 Scoring & Reference Comparison
# ==============================================================================
import numpy as np
import pandas as pd

df_preds = pd.read_parquet(PREDICTIONS_PATH)

def score_standard_best_of_4_row(row) -> float:
    c_score = float(row["chosen_scores"][0])
    r_scores = [float(x) for x in row["rejected_scores"]]
    max_r = max(r_scores)
    if c_score > max_r + 1e-6:
        return 1.0
    elif abs(c_score - max_r) <= 1e-6:
        num_tied_r = sum(1 for r in r_scores if abs(r - c_score) <= 1e-6)
        return 1.0 / float(1 + num_tied_r)
    return 0.0

def score_ties_subset_row(row) -> float:
    c_scores = [float(x) for x in row["chosen_scores"]]
    r_scores = [float(x) for x in row["rejected_scores"]]
    min_c, max_c, max_r = min(c_scores), max(c_scores), max(r_scores)
    all_correct_above_incorrect = 1.0 if (min_c > max_r) else 0.0
    sep_margin = min_c - max_r
    internal_spread = max_c - min_c
    margin_consistent = 1.0 if (all_correct_above_incorrect == 1.0 and sep_margin > internal_spread) else 0.0
    return 0.5 * all_correct_above_incorrect + 0.5 * margin_consistent

df_preds["rb2_score"] = [
    score_ties_subset_row(r) if str(r["subset"]).lower() == "ties" else score_standard_best_of_4_row(r)
    for _, r in df_preds.iterrows()
]

raw_domain_means = (df_preds.groupby("subset")["rb2_score"].mean() * 100.0).to_dict()
RB2_DOMAINS = ["Factuality", "Precise IF", "Math", "Safety", "Focus", "Ties"]
domain_scores = {dom: float(raw_domain_means.get(dom, 0.0)) for dom in RB2_DOMAINS}
overall_rb2_score = float(np.mean(list(domain_scores.values())))

print("=" * 85)
print("REWARDBENCH 2 EVALUATION SUMMARY: Stellar-Qwen3-8B-ARM")
print("=" * 85)
for dom in RB2_DOMAINS:
    print(f"  {dom:<22s}: {domain_scores[dom]:>6.2f}%")
print("-" * 85)
print(f"  {'OVERALL MACRO-AVERAGE':<22s}: {overall_rb2_score:>6.2f}%  (Random Baseline = 25.00%)")
print("=" * 85)

rb2_reference = [
    {"Model": "Skywork/Skywork-Reward-V2-Llama-3.1-8B", "Type": "8B Classifier (26M pairs)", "Overall": 84.13, "Factuality": 84.6, "Precise IF": 66.2, "Math": 77.6, "Safety": 96.7, "Focus": 98.4, "Ties": 81.2},
    {"Model": "ContextualAI/LMUnit-qwen2.5-72b", "Type": "72B Generative", "Overall": 82.08, "Factuality": 87.2, "Precise IF": 54.4, "Math": 72.7, "Safety": 91.3, "Focus": 96.8, "Ties": 90.1},
    {"Model": "Skywork/Skywork-Reward-V2-Qwen3-8B", "Type": "8B Classifier (26M pairs)", "Overall": 78.37, "Factuality": 79.9, "Precise IF": 50.0, "Math": 77.0, "Safety": 94.0, "Focus": 96.4, "Ties": 72.9},
    {"Model": "infly/INF-ORM-Llama3.1-70B", "Type": "70B Classifier", "Overall": 76.48, "Factuality": 74.1, "Precise IF": 41.9, "Math": 69.9, "Safety": 96.4, "Focus": 90.3, "Ties": 86.2},
    {"Model": "allenai/Llama-3.1-70B-Instruct-RM-RB2", "Type": "70B Classifier", "Overall": 76.06, "Factuality": 81.3, "Precise IF": 41.9, "Math": 69.9, "Safety": 88.4, "Focus": 86.5, "Ties": 88.3},
    {"Model": "Stellar-Qwen3-8B-ARM (Ours)", "Type": "8B Causal DAG (7.5k pairs)", "Overall": round(overall_rb2_score, 2), **{d: round(domain_scores[d], 1) for d in RB2_DOMAINS}},
    {"Model": "LxzGordon/URM-LLaMa-3.1-8B", "Type": "8B Classifier", "Overall": 73.94, "Factuality": 68.8, "Precise IF": 45.0, "Math": 63.9, "Safety": 91.8, "Focus": 97.6, "Ties": 76.5},
    {"Model": "openai/gpt-4.1-2025-04-14", "Type": "Proprietary Judge", "Overall": 72.32, "Factuality": 82.9, "Precise IF": 39.7, "Math": 65.2, "Safety": 87.3, "Focus": 73.4, "Ties": 85.4},
    {"Model": "anthropic/claude-sonnet-4-20250514", "Type": "Proprietary Judge", "Overall": 71.17, "Factuality": 76.1, "Precise IF": 35.9, "Math": 70.5, "Safety": 89.1, "Focus": 76.0, "Ties": 79.4},
    {"Model": "RLHFlow/ArmoRM-Llama3-8B-v0.1", "Type": "8B MoE RM", "Overall": 66.46, "Factuality": 65.7, "Precise IF": 41.9, "Math": 66.1, "Safety": 82.2, "Focus": 76.6, "Ties": 66.3},
    {"Model": "openai/gpt-4o-2024-08-06", "Type": "Proprietary Judge", "Overall": 64.93, "Factuality": 56.8, "Precise IF": 33.1, "Math": 62.3, "Safety": 86.2, "Focus": 72.9, "Ties": 78.2},
]

df_rb2_lb = pd.DataFrame(rb2_reference).sort_values(by="Overall", ascending=False).reset_index(drop=True)
df_rb2_lb.to_csv(LEADERBOARD_CSV_PATH, index=False)

print("\nREWARDBENCH 2 REFERENCE COMPARISON")
print("=" * 115)
print(df_rb2_lb.to_markdown(index=False))
print("=" * 115)

REWARDBENCH 2 EVALUATION SUMMARY: Stellar-Qwen3-8B-ARM
  Factuality            :  66.95%
  Precise IF            :  45.00%
  Math                  :  65.03%
  Safety                :  93.33%
  Focus                 :  96.97%
  Ties                  :  77.94%
-------------------------------------------------------------------------------------
  OVERALL MACRO-AVERAGE :  74.20%  (Random Baseline = 25.00%)

REWARDBENCH 2 REFERENCE COMPARISON
| Model                                  | Type                       |   Overall |   Factuality |   Precise IF |   Math |   Safety |   Focus |   Ties |
|:---------------------------------------|:---------------------------|----------:|-------------:|-------------:|-------:|---------:|--------:|-------:|
| Skywork/Skywork-Reward-V2-Llama-3.1-8B | 8B Classifier (26M pairs)  |     84.13 |         84.6 |         66.2 |   77.6 |     96.7 |    98.4 |   81.2 |
| ContextualAI/LMUnit-qwen2.5-72b        | 72B Generative             |     82.08 |         87.2 |

In [6]:
# ==============================================================================
# CELL 5: Export RewardBench 2 Comparison Figure
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.0), dpi=300, gridspec_kw={"width_ratios": [1.3, 1.0]})

df_plot = df_rb2_lb.sort_values(by="Overall", ascending=True)
models = [m.split("/")[-1] for m in df_plot["Model"]]
overalls = df_plot["Overall"].tolist()
bar_colors = ["#d95f02" if "Stellar" in m else ("#4a6fe3" if "8B" in m else "#8c8c8c") for m in df_plot["Model"]]

bars = ax1.barh(models, overalls, color=bar_colors, height=0.65, edgecolor="black", linewidth=0.7)
ax1.axvline(25.0, color="red", linestyle=":", linewidth=1.2, label="Random Baseline (25%)")
ax1.set_xlim(20, 95)
ax1.set_xlabel("Overall Macro-Average (%)", fontsize=11, fontweight="bold")
ax1.set_title("A. RewardBench 2 Reference Comparison", fontsize=11, fontweight="bold")
ax1.grid(axis="x", linestyle="--", alpha=0.4)
ax1.legend(loc="lower right", fontsize=9)

for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.6, bar.get_y() + bar.get_height() / 2, f"{w:.2f}%", va="center", ha="left", fontsize=8.5, fontweight="bold")

x = np.arange(len(RB2_DOMAINS))
width = 0.26
ours_row = df_rb2_lb[df_rb2_lb["Model"].str.contains("Stellar")].iloc[0]
sky_row  = df_rb2_lb[df_rb2_lb["Model"].str.contains("Skywork-Reward-V2-Qwen3-8B")].iloc[0]
armo_row = df_rb2_lb[df_rb2_lb["Model"].str.contains("ArmoRM")].iloc[0]

ax2.bar(x - width, [ours_row[d] for d in RB2_DOMAINS], width, label="Stellar-Qwen3-8B-ARM (7.5k pairs)", color="#d95f02", edgecolor="black")
ax2.bar(x,         [sky_row[d]  for d in RB2_DOMAINS], width, label="Skywork-V2-Qwen3-8B (26M pairs)", color="#4a6fe3", edgecolor="black")
ax2.bar(x + width, [armo_row[d] for d in RB2_DOMAINS], width, label="ArmoRM-Llama3-8B-v0.1", color="#8c8c8c", edgecolor="black")

ax2.set_xticks(x)
ax2.set_xticklabels(RB2_DOMAINS, rotation=20, ha="right", fontsize=10, fontweight="bold")
ax2.set_ylim(0, 108)
ax2.set_ylabel("Domain Score (%)", fontsize=11, fontweight="bold")
ax2.set_title("B. Per-Domain Comparison (8B Models)", fontsize=11, fontweight="bold")
ax2.legend(loc="upper left", fontsize=8.5)
ax2.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
fig.savefig(LEADERBOARD_PNG_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Image(filename=LEADERBOARD_PNG_PATH))

### Comparison Figure

![Stellar-Qwen3-8B-ARM — RewardBench 2 Evaluation](assets/rewardbench2_comparison.png)